## Structured Output:

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested strucutes.

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")  # reasoning model

d:\LangChain_hands_on\Langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002749E7A8EC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002749E7A9BE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description= "The title of the movie")
    year : int = Field(description= "this year the movie was released")
    director : str = Field(description="The directory of the movie")
    rating : float = Field(description="The movie's rating out of 10")

In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002749E7A8EC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002749E7A9BE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'The directory of the movie', 'type': 'string'}, 'rating': {'description': "The movie's rating out of 10", 'type': 'number'}}, 'required': ['title', 'year',

In [6]:
model.invoke("provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 science fiction action film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, who plays Dom Cobb. The movie\'s title, "Inception," refers to the concept of planting an idea in someone\'s subconscious, which is the main plot idea.\n\nThe story is about a thief who enters people\'s dreams to steal secrets. The main character, Cobb, is known as a thief who uses a special technique to extract information from targets while they\'re dreaming. He\'s offered a chance to have his criminal history erased by performing the reverse: planting an idea into someone\'s mind. This process is called "inception." The challenge is that it\'s much harder to plant an idea than to steal one because the subconscious is more resistant to being influenced.\n\nThe film explores different layers of dreams within dreams, each with different

In [7]:
model_with_structure.invoke("provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    '''A movie with details'''
    title : str = Field(description= "The title of the movie")
    year : int = Field(description= "this year the movie was released")
    director : str = Field(description="The directory of the movie")
    rating : float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw= True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to make sure I have all those details. I remember Inception was directed by Christopher Nolan. The release year was 2010. The rating is a bit trickier, but I think it's around 8.8 on IMDb. Let me confirm that. Yes, 8.8 out of 10. So I should structure the function call with those parameters. Make sure the JSON is correctly formatted with the required fields. No need to include any extra information. Just the title, year, director, and rating. Double-check the types: title and director are strings, year is an integer, rating is a number. Everything looks good. Time to put it all together in the tool_call format.\n", 'tool_calls': [{'id': 'y69fxc9e0', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","

### Nested Structure

In [9]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description= "Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide detials about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action'], budget=160.0)

## TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validaiton.

In [10]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    '''A movie with details'''
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]
    

In [11]:
model_with_typeddict=  model.with_structured_output(MovieDict)

response= model_with_typeddict.invoke("Provide the details about the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

## DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass